# AttnFuse Quickstart

Five-minute tour of the things AttnFuse can do that PyTorch's built-in
attention APIs can't. Each cell is a complete runnable example. **Requires a CUDA GPU**
(any sm_86+ — 3090, A100, L4, H100, etc.).

Install:
```bash
pip install torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
pip install triton==3.1.0
pip install -e .
```

## Outline
1. Dense / causal / sliding-window attention in 5 lines each
2. **Fused RoPE** — the thing `flex_attention` cannot do
3. Grouped-Query Attention (Llama-3 / Mistral / Falcon geometries)
4. KV-cache autoregressive decoding (Flash Decoding kicks in automatically)
5. **Training** via `torch.autograd`
6. **Block-sparse attention** (BigBird-style with training support)
7. HuggingFace integration: `attn_implementation="attnfuse"`

In [ ]:
import torch
import attnfuse as af
torch.manual_seed(0)

print('GPU      :', torch.cuda.get_device_name(0))
print('AttnFuse :', af.__version__)
print()

B, H, N, D = 2, 12, 1024, 64
Q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
K = torch.randn_like(Q)
V = torch.randn_like(Q)
print(f'Q shape : {tuple(Q.shape)}  dtype : {Q.dtype}')

## 1. Dense / causal / sliding-window in five lines each

In [ ]:
@af.attention
def dense(Q, K, V):
    return af.softmax(af.scaled_dot_product(Q, K)) @ V

@af.attention
def causal(Q, K, V):
    return af.softmax(af.causal(af.scaled_dot_product(Q, K))) @ V

@af.attention
def mistral_local(Q, K, V):
    s = af.sliding_window(af.scaled_dot_product(Q, K), window_size=256)
    return af.softmax(s) @ V

for fn in (dense, causal, mistral_local):
    out = fn(Q, K, V)
    print(f'  {fn.__name__:14}  -> {tuple(out.shape)}')

## 2. Fused RoPE — what `flex_attention` *cannot* do

`flex_attention`'s `score_mod` hook fires **after** $QK^\top$. RoPE needs to rotate $Q$ and $K$
**before** the matmul, so `flex_attention` falls back to two extra host-side kernels +
an HBM round-trip. AttnFuse rotates inside the inner loop.

In [ ]:
from attnfuse.rope_utils import build_rope_cache
cos, sin = build_rope_cache(N, D, device='cuda', dtype=torch.float16)

@af.attention
def llama_attn(Q, K, V):
    s = af.rope(Q, K)                          # rotate Q, K inside the kernel
    s = af.causal(s)
    return af.softmax(s) @ V

out = llama_attn(Q, K, V, cos=cos, sin=sin)
print(f'fused RoPE -> {tuple(out.shape)}, dtype={out.dtype}')
print(f'\nBenchmark vs flex_attention (causal+RoPE @ N=4096): up to 2.10x speedup.')

## 3. Grouped-Query Attention (Llama-3-8B geometry)

In [ ]:
# Llama-3-8B uses H_q=32, H_kv=8, D=128 (group_size = 4).
B_, N_, D_ = 1, 512, 128
Qg = torch.randn(B_, 32, N_, D_, device='cuda', dtype=torch.float16)
Kg = torch.randn(B_, 8,  N_, D_, device='cuda', dtype=torch.float16)   # 4x fewer KV heads
Vg = torch.randn(B_, 8,  N_, D_, device='cuda', dtype=torch.float16)

out = causal(Qg, Kg, Vg)
print(f'GQA causal -> {tuple(out.shape)}')
print('   (AttnFuse natively broadcasts via the GROUP_SIZE constexpr -- no repeat_kv copy.)')

## 4. KV-cache autoregressive decoding (Flash Decoding fast path)

When `Q.N == 1` and the K cache is large, AttnFuse dispatches automatically to
a two-phase split-K kernel. At cache=32768 on Llama-3-70B geometry, this gives a
**17× speedup** vs the standard kernel and **1.06×** over flex_attention.

In [ ]:
# One new token attending to a large KV cache.
Q1 = torch.randn(1, 32,    1, 128, device='cuda', dtype=torch.float16)   # one query token
Kc = torch.randn(1,  8, 4096, 128, device='cuda', dtype=torch.float16)   # cache
Vc = torch.randn(1,  8, 4096, 128, device='cuda', dtype=torch.float16)

out = dense(Q1, Kc, Vc)
print(f'decode step -> {tuple(out.shape)}  (Flash Decoding auto-dispatched)')

## 5. Training — `torch.autograd` works automatically

In [ ]:
Q.requires_grad_(); K.requires_grad_(); V.requires_grad_()
out = causal(Q, K, V)
loss = out.sum()
loss.backward()              # AttnFuse backward kernels run
print(f'dQ shape: {tuple(Q.grad.shape)}  |dQ|_max = {Q.grad.abs().max():.4f}')
print(f'dK shape: {tuple(K.grad.shape)}  |dK|_max = {K.grad.abs().max():.4f}')
print(f'dV shape: {tuple(V.grad.shape)}  |dV|_max = {V.grad.abs().max():.4f}')

## 6. Block-sparse attention (BigBird-style + training)

Build a CSR-style `BlockMask` from a Python predicate. Both forward and backward
kernels iterate only over active blocks — genuine sub-quadratic FLOPs.

In [ ]:
# BigBird-style: global first row + global first col + local window of width 1.
def bigbird(q, kv):
    qb = q // 64; kb = kv // 64
    return (qb == 0) | (kb == 0) | ((qb - kb).abs() <= 1)

mask = af.create_block_mask(bigbird, Q_LEN=N, KV_LEN=N, BLOCK_M=64, BLOCK_N=64)
print(f'BlockMask: {mask.n_active} / {mask.n_q_blocks * mask.n_kv_blocks} active blocks '
      f'({100*mask.n_active/(mask.n_q_blocks*mask.n_kv_blocks):.1f}%)')

@af.attention
def bigbird_attn(Q, K, V):
    s = af.scaled_dot_product(Q, K)
    s = af.block_sparse(s)
    return af.softmax(s) @ V

Q_, K_, V_ = (t.detach().requires_grad_() for t in (Q, K, V))
out = bigbird_attn(Q_, K_, V_, block_mask=mask)
out.sum().backward()
print(f'\nblock-sparse forward + backward, both sub-quadratic in active fraction.')

## 7. HuggingFace integration: drop into any Llama-class model

```python
import attnfuse.integrations.hf  # registers `attnfuse` with HF
import transformers

model = transformers.LlamaForCausalLM.from_pretrained(
    'meta-llama/Llama-3-8B',
    attn_implementation='attnfuse',
    torch_dtype=torch.float16,
)
# That's it -- AttnFuse runs every attention layer.
```

Measured at Llama-3-8B-class dimensions on a 3090: AttnFuse is **1.06× of
PyTorch SDPA's hand-tuned CUDA on a full training step** (forward + backward
+ optimizer.step()). `flex_attention` OOMs at HEAD_DIM=128 on the 3090; AttnFuse
is the only Triton-based attention compiler that runs Llama-3-8B training
on consumer-grade Ampere.

## Where to read more

- **`docs/AttnFuse_paper.tex`** — MLSys-class paper draft with all numbers.
- **`PROGRESS.md`** — what's been built and why, with measured impact for each
  contribution.
- **`results/`** — committed CSVs from every benchmark (forward, backward, KV-cache,
  block-sparse, HF Llama-3, H100 sweep, roofline).